# Cogniland Sweep Analysis

Analysis of the hyperparameter sweep over **time_penalty**, **lambda_p**, and **difficulty**.

Sweep grid:
- `time_penalty`: 0.01, 0.05, 0.10, 0.20
- `lambda_p`: 0.2, 0.5, 1.0
- `difficulty`: default (1×), half (0.5×), fifth (0.2× resource drain)

Total: 4 × 3 × 3 = 36 runs

**Note on metrics:** Val metrics (success rate, behavioral) are pulled directly from WandB.
Test success rate was not logged during training — it is computed here by running each checkpoint against the test map split.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import wandb
from IPython.display import display, HTML

sys.path.insert(0, str(Path(".").resolve()))

sns.set_theme(style="whitegrid", context="notebook", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120

SWEEP_RUN_IDS = [
    "02agthx8","0fjc3yvj","0ot2ghk8","27xcjita","2np83te6","3psg1lr8",
    "3x0j9c69","4j0mo0ws","4yhibzyb","55po9bfo","6s3d2u1d","7nfloswx",
    "8sl8z1ma","96x12583","99v68sf7","9xqfrhqc","a948o8tq","ahk1o7q9",
    "bpkulemc","bw90jdtn","e2ijy7u4","e3c0itct","eyeuudwv","f665rxcq",
    "ie0scu8d","jzpgle7e","leo31vpb","lx9pzxdr","m9ssmoyp","q0cbeblk",
    "qd9b25p9","queyzb76","sbd3ct4q","v18v9a9d","vi60lv82","zwpxpm5l",
]

ARTIFACTS_DIR = Path("artifacts_sweep")
CACHE_PATH    = Path("data/sweep_test_results.csv")

## 1. Fetch sweep runs from W&B

In [ ]:
api = wandb.Api()

# Fetch by explicit run IDs — tag-based filter returns stale/empty configs
print("Fetching run metadata from W&B …")
wb_runs = {}
for rid in SWEEP_RUN_IDS:
    wb_runs[rid] = api.run(f"crusoe/cogniland/{rid}")
print(f"Fetched {len(wb_runs)} runs")

## 2. Build DataFrame (last-iteration metrics)

In [ ]:
def _tag_float(tags, prefix):
    """Extract float from a tag like 'tp_0.01' given prefix 'tp_'."""
    for tag in tags:
        if tag.startswith(prefix):
            try:
                return float(tag[len(prefix):])
            except ValueError:
                pass
    return None

DIFFICULTY_LABELS = {
    "default": "default (1×)",
    "half":    "half (0.5×)",
    "fifth":   "fifth (0.2×)",
}
DIFFICULTY_NUMERIC = {
    "default (1×)":  1.0,
    "half (0.5×)":   0.5,
    "fifth (0.2×)":  0.2,
}

# Val metrics available in W&B summary
VAL_METRICS = {
    "val_det/env/success_rate":        "val_success_rate",
    "val_det/env/directness_mean":     "val_directness",
    "val_det/env/exploration_mean":    "val_exploration",
    "val_det/env/risk_exposure_mean":  "val_risk_exposure",
}

records = []
for run in wb_runs.values():
    tags = run.tags
    s = run.summary

    # Decode difficulty from tags: "diff_default" → "default", "diff_half" → "half", …
    diff_raw = next((t[5:] for t in tags if t.startswith("diff_")), None)

    row = {
        "run_id":       run.id,
        "run_name":     run.name,
        "time_penalty": _tag_float(tags, "tp_"),
        "lambda_p":     _tag_float(tags, "lp_"),
        "difficulty":   DIFFICULTY_LABELS.get(diff_raw, diff_raw or "unknown"),
    }
    for wb_key, col in VAL_METRICS.items():
        row[col] = s.get(wb_key)

    records.append(row)

df = pd.DataFrame(records)
# Test metrics populated by local evaluation below
for c in ["test_success_rate", "test_directness", "test_exploration", "test_risk_exposure"]:
    df[c] = float("nan")

print(f"Built DataFrame: {len(df)} runs")
df.head()

In [ ]:
# Sanity check: unique values per sweep axis
for col in ["time_penalty", "lambda_p", "difficulty"]:
    print(f"{col:>15}: {sorted(df[col].dropna().unique())}")
print(f"\nTotal runs: {len(df)}, val_success_rate non-null: {df['val_success_rate'].notna().sum()}")

## 2.5 Local Test Evaluation

Test metrics were not logged during training. Here we load each `ckpt_best.pt` and evaluate it on the held-out test maps using the current default environment config. Results are cached to `data/sweep_test_results.csv`.

In [ ]:
from omegaconf import OmegaConf
from cogniland.env.types import EnvConfig
from cogniland.env.wrappers import BatchedIslandEnv
from cogniland.env.dataset import MapDataset
from cogniland.models import build_model
from cogniland.eval import CognilandSummarizer, EvalRunner

device_str = ("cuda" if torch.cuda.is_available()
              else ("mps" if torch.backends.mps.is_available() else "cpu"))
device = torch.device(device_str)
print(f"Device: {device_str}")

# Build cfg from current config files (architecture is identical to sweep checkpoints)
_env_yaml   = OmegaConf.load("configs/env/default.yaml")
_model_yaml = OmegaConf.load("configs/models/ppo.yaml")
cfg = OmegaConf.create({
    "device": device_str,
    "env": OmegaConf.to_container(_env_yaml, resolve=True),
    "models": OmegaConf.to_container(_model_yaml, resolve=True),
})

# Load dataset — shared test split for all runs
dataset = MapDataset.load(cfg.models["training"]["dataset"]["path"])
n_test_eps = len(dataset.test_maps)
env_config = EnvConfig.from_hydra(cfg)
print(f"Test maps: {n_test_eps}")

summarizer = CognilandSummarizer()

if CACHE_PATH.exists():
    print(f"Loading cached results from {CACHE_PATH}")
    test_df = pd.read_csv(CACHE_PATH)
else:
    # Build a fresh eval env for each run (reset reuses the same map pool)
    test_records = []
    missing = []
    for i, run_id in enumerate(SWEEP_RUN_IDS):
        ckpt_path = ARTIFACTS_DIR / run_id / "ckpt_best.pt"
        if not ckpt_path.exists():
            print(f"  [{i+1:2d}/{len(SWEEP_RUN_IDS)}] {run_id}: MISSING — skip")
            missing.append(run_id)
            continue

        eval_env = BatchedIslandEnv(
            env_config,
            num_envs=n_test_eps,
            world_maps=dataset.test_maps,
        )
        runner = EvalRunner(eval_env, env_config, device_str)

        model = build_model(cfg)
        ckpt  = torch.load(ckpt_path, map_location=device, weights_only=False)
        model.model.load_state_dict(ckpt["model_state_dict"])
        model.model.to(device)
        model.model.eval()

        result = runner.run(
            policy_fn=lambda obs: model.get_deterministic_action(obs),
            n_episodes=n_test_eps,
            mode="det",
            split="test",
            global_step=int(ckpt.get("step", 0)),
        )
        m = summarizer.scalar_metrics(result)

        test_records.append({
            "run_id":             run_id,
            "test_success_rate":  m["test_det/env/success_rate"],
            "test_directness":    m["test_det/env/directness_mean"],
            "test_exploration":   m["test_det/env/exploration_mean"],
            "test_risk_exposure": m["test_det/env/risk_exposure_mean"],
        })
        sr = m["test_det/env/success_rate"]
        print(f"  [{i+1:2d}/{len(SWEEP_RUN_IDS)}] {run_id}: success_rate={sr:.3f}")

    if missing:
        print(f"\nWarning: {len(missing)} checkpoints missing: {missing}")

    test_df = pd.DataFrame(test_records)
    CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    test_df.to_csv(CACHE_PATH, index=False)
    print(f"\nSaved to {CACHE_PATH}")

test_df.head()

In [ ]:
# Merge test metrics into main DataFrame
df = df.drop(columns=["test_success_rate", "test_directness", "test_exploration", "test_risk_exposure"])
df = df.merge(test_df, on="run_id", how="left")
n_test = df["test_success_rate"].notna().sum()
print(f"DataFrame: {len(df)} rows, {n_test}/{len(df)} with test metrics")
df[["run_id", "time_penalty", "lambda_p", "difficulty",
    "val_success_rate", "test_success_rate"]].head(10)

## 3. Sortable results table

Interactive table — click column headers to sort by test success rate, behavioral metrics, etc.

In [ ]:
TABLE_COLS = [
    "run_id", "time_penalty", "lambda_p", "difficulty",
    "test_success_rate", "val_success_rate",
    "test_directness", "test_exploration", "test_risk_exposure",
    "val_directness", "val_exploration", "val_risk_exposure",
]

df_table = (
    df[TABLE_COLS]
    .sort_values("test_success_rate", ascending=False)
    .reset_index(drop=True)
)

try:
    from itables import show
    show(
        df_table,
        paging=False,
        columnDefs=[{"className": "dt-center", "targets": "_all"}],
    )
except ImportError:
    float_cols = df_table.select_dtypes(include="number").columns
    styled = (
        df_table.style
        .format({c: "{:.4f}" for c in float_cols})
        .background_gradient(
            subset=["test_success_rate"], cmap="RdYlGn"
        )
    )
    display(styled)
    print("\n(Tip: install `itables` for interactive sorting — pip install itables)")

## 4. Effect of sweep variables on test success rate

Line plots faceted by **difficulty** to show how each reward parameter influences test performance.

### 4a. `time_penalty` vs test success rate

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)

for ax, (diff, grp) in zip(axes, df.groupby("difficulty")):
    for lp, sub in grp.groupby("lambda_p"):
        sub_sorted = sub.sort_values("time_penalty")
        ax.plot(
            sub_sorted["time_penalty"],
            sub_sorted["test_success_rate"],
            marker="o", label=f"λ_p={lp}",
        )
    ax.set_title(f"Difficulty: {diff}")
    ax.set_xlabel("time_penalty")
    ax.legend(fontsize=9)

axes[0].set_ylabel("Test Success Rate")
fig.suptitle("Effect of time_penalty on Test Success Rate", fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

### 4b. `lambda_p` vs test success rate

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)

for ax, (diff, grp) in zip(axes, df.groupby("difficulty")):
    for tp, sub in grp.groupby("time_penalty"):
        sub_sorted = sub.sort_values("lambda_p")
        ax.plot(
            sub_sorted["lambda_p"],
            sub_sorted["test_success_rate"],
            marker="o", label=f"tp={tp}",
        )
    ax.set_title(f"Difficulty: {diff}")
    ax.set_xlabel("lambda_p")
    ax.legend(fontsize=9)

axes[0].set_ylabel("Test Success Rate")
fig.suptitle("Effect of λ_p on Test Success Rate", fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

### 4c. `difficulty` vs test success rate

In [ ]:
DIFF_ORDER = ["default (1×)", "half (0.5×)", "fifth (0.2×)"]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)

for ax, (lp, grp) in zip(axes, df.groupby("lambda_p")):
    for tp, sub in grp.groupby("time_penalty"):
        sub = sub.set_index("difficulty").reindex(DIFF_ORDER).reset_index()
        ax.plot(
            sub["difficulty"],
            sub["test_success_rate"],
            marker="o", label=f"tp={tp}",
        )
    ax.set_title(f"λ_p = {lp}")
    ax.set_xlabel("difficulty")
    ax.tick_params(axis="x", rotation=20)
    ax.legend(fontsize=9)

axes[0].set_ylabel("Test Success Rate")
fig.suptitle("Effect of Difficulty on Test Success Rate", fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

## 5. Marginal effect (averaged over other axes)

Each sweep variable averaged across the other two for a clearer aggregate trend.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# time_penalty
agg = df.groupby("time_penalty")["test_success_rate"].agg(["mean", "std"]).reset_index()
axes[0].errorbar(agg["time_penalty"], agg["mean"], yerr=agg["std"], marker="o", capsize=4)
axes[0].set_xlabel("time_penalty")
axes[0].set_ylabel("Test Success Rate")
axes[0].set_title("Marginal effect of time_penalty")

# lambda_p
agg = df.groupby("lambda_p")["test_success_rate"].agg(["mean", "std"]).reset_index()
axes[1].errorbar(agg["lambda_p"], agg["mean"], yerr=agg["std"], marker="o", capsize=4)
axes[1].set_xlabel("lambda_p")
axes[1].set_title("Marginal effect of λ_p")

# difficulty
agg = df.groupby("difficulty")["test_success_rate"].agg(["mean", "std"]).reindex(DIFF_ORDER).reset_index()
axes[2].errorbar(range(len(agg)), agg["mean"], yerr=agg["std"], marker="o", capsize=4)
axes[2].set_xticks(range(len(agg)))
axes[2].set_xticklabels(agg["difficulty"], rotation=20)
axes[2].set_xlabel("difficulty")
axes[2].set_title("Marginal effect of difficulty")

fig.suptitle("Marginal Effects on Test Success Rate (mean ± std)", fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

## 6. Correlation matrix: hyperparameters × behavioral metrics

Pearson correlation between the swept hyperparameters and behavioral metrics in both **validation** and **test** (deterministic policy).

Difficulty is encoded numerically as the terrain resource-cost multiplier: default=1.0, half=0.5, fifth=0.2.

In [ ]:
df_corr = df.copy()
df_corr["difficulty_num"] = df_corr["difficulty"].map(DIFFICULTY_NUMERIC)

hp_cols = ["time_penalty", "lambda_p", "difficulty_num"]
metric_cols = [
    "val_success_rate", "val_directness", "val_exploration", "val_risk_exposure",
    "test_success_rate", "test_directness", "test_exploration", "test_risk_exposure",
]

corr = df_corr[hp_cols + metric_cols].corr()
corr_subset = corr.loc[hp_cols, metric_cols]

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(
    corr_subset,
    annot=True, fmt=".2f",
    cmap="RdBu_r", center=0, vmin=-1, vmax=1,
    linewidths=0.5, ax=ax,
)
ax.set_title("Hyperparameters × Behavioral Metrics (Pearson r)", fontweight="bold")
ax.set_yticklabels(["time_penalty", "λ_p", "difficulty\n(cost multiplier)"], rotation=0)
fig.tight_layout()
plt.show()

### Full correlation matrix (all variables)

In [ ]:
all_cols = hp_cols + metric_cols
full_corr = df_corr[all_cols].corr()

labels = [
    "time_penalty", "λ_p", "difficulty",
    "val SR", "val direct.", "val explor.", "val risk",
    "test SR", "test direct.", "test explor.", "test risk",
]

mask = np.triu(np.ones_like(full_corr, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    full_corr,
    mask=mask,
    annot=True, fmt=".2f",
    cmap="RdBu_r", center=0, vmin=-1, vmax=1,
    linewidths=0.5, ax=ax,
    xticklabels=labels, yticklabels=labels,
)
ax.set_title("Full Correlation Matrix", fontweight="bold")
fig.tight_layout()
plt.show()

## 7. Behavioral metrics by difficulty (val & test)

In [ ]:
behavioral = ["directness", "exploration", "risk_exposure"]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for ax, metric in zip(axes, behavioral):
    melted = pd.melt(
        df,
        id_vars=["difficulty"],
        value_vars=[f"val_{metric}", f"test_{metric}"],
        var_name="split", value_name=metric,
    )
    melted["split"] = melted["split"].str.replace(f"_{metric}", "")
    sns.boxplot(
        data=melted, x="difficulty", y=metric, hue="split",
        order=DIFF_ORDER, ax=ax, palette="Set2",
    )
    ax.set_title(metric.replace("_", " ").title())
    ax.set_xlabel("difficulty")
    ax.tick_params(axis="x", rotation=20)

fig.suptitle("Behavioral Metrics by Difficulty (val vs test)", fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()